# Liu2024 — Paper-faithful TWFB-DGFMDM (19 bands × 7 windows + LTSA), with cache + leaky/honest

The **paper's Table-3 method**, exactly as described, wired into the v2 machinery (covariance cache + logging):

1. **Step 1** — 7 time windows (0–1 … 3–4 s) × **19 overlapping 4-Hz bands** (8–12, 9–13, … 26–30) = 133 views.
2. **Step 3** — per-trial covariance for each view (cached and reusable).
3. **Step 4** — **LTSA** (`sklearn LocallyLinearEmbedding(method="ltsa")`) applied **after the covariance**:
   covariance → Riemannian tangent space → (PCA pre-reduce) → **LTSA** → reduced features.
4. **Step 5** — discriminant + nearest-mean classifier on the reduced features (`lda_ltsa`); `fgmdm` available
   as the published-code reading (covariance → FgMDM, no LTSA).

**Two protocols** over the 133 views (toggle in CONFIG):
- `twfb_leaky`  — view chosen by accuracy on the **test** set (reproduces the published-style inflation).
- `twfb_honest` — view chosen by **inner CV on the training split only**, scored once on the held-out test fold.
- `perband_fixed` — each band, full window, no selection (a no-cherry-picking floor).

> **What is cached vs not.** Only the per-trial **covariances** are cached (they are config-determined and
> per-trial — see `COVARIANCE_CACHE_README.md`). **LTSA and the classifier are fit per fold on the training data
> only** and are *not* cached — that is what keeps the honest protocol leakage-free.

> Logging/artifacts mirror `sjepa_prelocal`. **Edit the single CONFIG cell (or apply a sweep) and re-run.**
> The 19×7 grid + LTSA is heavier than the shipped-code notebook — start with the `PV2_canonical_quick` sweep.

# 1. Setup

In [ ]:
import os, sys, glob, json, time, random, hashlib, builtins, platform, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import butter, iirnotch, lfilter, filtfilt
from scipy import stats as sp_stats

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.metrics import accuracy_score, balanced_accuracy_score

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as e:
    HAVE_MPL = False; print("matplotlib unavailable:", e)

from pyriemann.classification import FgMDM     # == MATLAB fgmdm (DGFMDM)
from pyriemann.tangentspace import TangentSpace

warnings.filterwarnings("ignore")
print("python", platform.python_version(), "| numpy", np.__version__)
import pyriemann; print("pyriemann", pyriemann.__version__)

# 2. CONFIG  *(edit this one cell, or apply a sweep JSON, then re-run)*

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    "experiment_name": "twfb_dgfmdm_paper_faithful_v2",
    "config_note":     "paper Table-3: 19 bands x 7 windows + LTSA; cache + leaky/honest",

    "source_roots": [
        str(WORKING_DIR.parent / "Liu2024_matlab_code" / "sourcedata"),
        str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    ],
    "artifact_dir":         str(WORKING_DIR / "artifacts" / "liu2024-twfb-dgfmdm-paper-faithful-v2"),
    "covariance_cache_dir": str(WORKING_DIR / "artifacts" / "covariance_cache"),
    "covariance_cache_mode": "auto",     # 'auto' | 'readonly' | 'rebuild' | 'off'

    "native_sfreq": 500,
    "channel_indices": list(range(0,17)) + list(range(18,30)),   # 29 ch, drop CPz
    "marker_channel_index": 32,
    "onset_marker_value": 2,
    "onset_plausible_range": [800, 1300],
    "onset_fallback_sample": 1003,
    "subjects_to_use": None,

    "preroll_samples": 800,
    "mi_window_samples": 2000,
    "notch_freq": 50.0, "notch_Q": 6.0,
    "butter_order": 2,
    "filter_phase": "zero",              # paper qualitative section uses zero-phase; 'causal' = MATLAB one-pass

    # ---- the paper's grid (Table 3) ----
    "freq_bands": [[8,12],[9,13],[10,14],[11,15],[12,16],[13,17],[14,18],[15,19],[16,20],
                   [17,21],[18,22],[19,23],[20,24],[21,25],[22,26],[23,27],[24,28],[25,29],[26,30]],
    "time_window_starts_s": [0.0,0.5,1.0,1.5,2.0,2.5,3.0],
    "time_window_len_s": 1.0,

    # ---- covariance ----
    "cov_trace_normalize": True,
    "cov_shrinkage": 0.10,

    # ---- Step 5 classifier ----
    "twfb_classifier": "lda_ltsa",       # 'lda_ltsa' (paper: TangentSpace->PCA->LTSA->LDA) | 'fgmdm' (DGFMDM on cov)
    "twfb_metric": "riemann",            # 'riemann' | 'logeuclid'

    # ---- Step 4 LTSA ----
    "use_ltsa": True,                    # False => ablation: TangentSpace->PCA->LDA (no LTSA)
    "ltsa_pre_pca": 20,                  # reduce 435-d tangent -> this before LTSA (stability); null to skip
    "ltsa_n_components": 6,
    "ltsa_n_neighbors": 10,
    "ltsa_reg": 1e-2,

    # ---- protocols ----
    "run_twfb_leaky":    True,
    "run_twfb_honest":   True,
    "run_perband_fixed": True,

    # ---- evaluation (heavier grid -> fewer repeats by default) ----
    "cv_scheme": "repeated_holdout",     # 'repeated_holdout' | 'kfold5'
    "n_repeats": 5,
    "train_size": 24, "test_size": 16,
    "honest_test_frac": 0.40,
    "n_splits": 5,
    "inner_folds": 3,
    "random_state": 2026,
}

BANDS = [tuple(b) for b in CONFIG["freq_bands"]]
FS    = CONFIG["native_sfreq"]
CH    = CONFIG["channel_indices"]
NCH   = len(CH)
WIN_STARTS = CONFIG["time_window_starts_s"]
print(f"channels={NCH}  bands={len(BANDS)}  windows={len(WIN_STARTS)}  TWxFB views={len(BANDS)*len(WIN_STARTS)}")
print(f"classifier={CONFIG['twfb_classifier']}  use_ltsa={CONFIG['use_ltsa']}  cache={CONFIG['covariance_cache_mode']}")

## 2.1 Logging, Run ID, Covariance-cache signature

In [ ]:
def _cov_signature(cfg):
    keys = ["native_sfreq","channel_indices","marker_channel_index","onset_marker_value",
            "onset_plausible_range","onset_fallback_sample","preroll_samples","mi_window_samples",
            "notch_freq","notch_Q","butter_order","filter_phase","freq_bands",
            "time_window_starts_s","time_window_len_s","cov_trace_normalize","cov_shrinkage"]
    subset = {k: cfg[k] for k in keys}
    h = hashlib.md5(json.dumps(subset, sort_keys=True, default=str).encode()).hexdigest()[:10]
    return h, subset

def create_run_id():
    h = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{datetime.now().strftime('%Y%m%d_%H%M')}_{h}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
COV_SIG, COV_SUBSET = _cov_signature(CONFIG)
COV_DIR = Path(CONFIG["covariance_cache_dir"]) / f"cov_{COV_SIG}"

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _w(stream, t):
    try: stream.write(t)
    except UnicodeEncodeError:
        enc=getattr(stream,"encoding",None) or "utf-8"; stream.write(t.encode(enc,"replace").decode(enc,"replace"))
def _tprint(*a, **k):
    sep=k.pop("sep"," "); end=k.pop("end","\n"); k.pop("flush",False); k.pop("file",None)
    msg=sep.join(str(x) for x in a); lead=len(msg)-len(msg.lstrip("\n")); body=msg[lead:]
    if lead: _w(sys.stdout,"\n"*lead); _w(_LOG,"\n"*lead)
    if body:
        s=f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {body}"; _w(sys.stdout,s+end); _w(_LOG,s+end)
    else: _w(sys.stdout,end); _w(_LOG,end)
builtins.print=_tprint
random.seed(CONFIG["random_state"]); np.random.seed(CONFIG["random_state"])
with open(ARTIFACT_DIR/"config.json","w") as f: json.dump(CONFIG,f,indent=2,default=str)
print("="*70)
print(f"Experiment: {CONFIG['experiment_name']}")
print(f"Note:       {CONFIG['config_note']}")
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Cov cache:  {COV_DIR}  (mode={CONFIG['covariance_cache_mode']}, sig={COV_SIG})")
print("="*70)

# 3. Data loading + MI onset

In [ ]:
def find_subject_files(roots):
    for root in roots:
        files = sorted(glob.glob(os.path.join(root, "sub-*", "sub-*_eeg.mat")))
        if files:
            print(f"using source root: {root}  ({len(files)} subjects)")
            return files
    raise FileNotFoundError(f"No sub-*_eeg.mat under any of: {roots}")

def load_subject_raw(path):
    m = sio.loadmat(path); eeg = m["eeg"][0,0]
    raw = np.asarray(eeg["rawdata"], dtype=np.float64)
    lab = np.asarray(eeg["label"]).ravel().astype(int)
    if set(np.unique(lab)).issubset({1,2}): lab = lab - 1
    mark = raw[:, CONFIG["marker_channel_index"], :]
    lo, hi = CONFIG["onset_plausible_range"]
    first = []
    for t in range(raw.shape[0]):
        idx = np.where(mark[t] == CONFIG["onset_marker_value"])[0]
        in_range = idx[(idx >= lo) & (idx <= hi)]
        first.append(int(in_range[0]) if in_range.size else (int(idx[0]) if idx.size else -1))
    plausible = [o for o in first if lo <= o <= hi]
    med = int(np.median(plausible)) if plausible else CONFIG["onset_fallback_sample"]
    onsets = [o if lo <= o <= hi else med for o in first]
    return raw, lab, np.asarray(onsets, dtype=int)
print("Loader defined.")

# 4. Covariance construction + reusable cache

In [ ]:
def _apply(b, a, x):
    return filtfilt(b, a, x, axis=1) if CONFIG["filter_phase"] == "zero" else lfilter(b, a, x, axis=1)

def _regularize(c):
    if CONFIG["cov_trace_normalize"]: c = c / np.trace(c)
    g = CONFIG["cov_shrinkage"]
    if g > 0:
        scale = (1.0/NCH) if CONFIG["cov_trace_normalize"] else (np.trace(c)/NCH)
        c = (1-g)*c + g*scale*np.eye(NCH)
    return c

def _band_segment(raw, onsets, bd):
    wo = CONFIG["notch_freq"]/(FS/2); bw = wo/CONFIG["notch_Q"]; nb, na = iirnotch(wo, wo/bw)
    bb, ba = butter(CONFIG["butter_order"], [bd[0]/(FS/2), bd[1]/(FS/2)], btype="band")
    pre = CONFIG["preroll_samples"]; L = CONFIG["mi_window_samples"]; n = raw.shape[0]
    out = np.zeros((n, NCH, L))
    for t in range(n):
        s0 = onsets[t] - pre
        seg = raw[t][:, s0:s0+pre+L][CH, :]
        seg = _apply(nb, na, seg); seg = _apply(bb, ba, seg)
        out[t] = seg[:, pre:pre+L]
    return out

def _cov(seg_slice):
    n = seg_slice.shape[0]; C = np.zeros((n, NCH, NCH))
    for t in range(n):
        X = seg_slice[t]; C[t] = _regularize(X @ X.T)
    return C

def build_subject_covariances(raw, onsets):
    views = {}; wlen = int(round(CONFIG["time_window_len_s"]*FS))
    for bd in BANDS:
        seg = _band_segment(raw, onsets, bd)
        views[f"fb__{bd[0]}_{bd[1]}"] = _cov(seg)
        for ws in WIN_STARTS:
            s = int(round(ws*FS)); views[f"tw__{bd[0]}_{bd[1]}__{ws}"] = _cov(seg[:, :, s:s+wlen])
    return views

def _write_manifest():
    COV_DIR.mkdir(parents=True, exist_ok=True); man = COV_DIR/"cache_manifest.json"
    if not man.exists():
        json.dump({"cov_signature": COV_SIG, "built_by": CONFIG["experiment_name"], "config_subset": COV_SUBSET,
                   "array_shape": "(n_trials, 29, 29)",
                   "keys": {"fb__{lo}_{hi}":"full 0-4 s covariance","tw__{lo}_{hi}__{wstart}":"1-s window covariance",
                            "y":"labels (0=left,1=right)","onsets":"MI onset sample","_cov_sig":"signature"},
                   "bands": CONFIG["freq_bands"], "windows": WIN_STARTS, "win_len_s": CONFIG["time_window_len_s"]},
                  open(man,"w"), indent=2, default=str)

def save_subject_cov(sid, views, y, onsets):
    COV_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(COV_DIR/f"sub-{sid:02d}_cov.npz",
                        y=np.asarray(y), onsets=np.asarray(onsets), _cov_sig=np.array(COV_SIG), **views)
    _write_manifest()

def load_subject_cov(sid):
    p = COV_DIR/f"sub-{sid:02d}_cov.npz"
    if not p.exists(): return None
    z = np.load(p, allow_pickle=False)
    if "_cov_sig" in z.files and str(z["_cov_sig"]) != COV_SIG:
        print(f"  WARN sub-{sid:02d}: cache signature mismatch")
    return {k: z[k] for k in z.files if k.startswith(("fb__","tw__"))}, z["y"], z["onsets"]

def get_subject_covariances(sid, path):
    mode = CONFIG["covariance_cache_mode"]
    if mode in ("auto","readonly"):
        c = load_subject_cov(sid)
        if c is not None: return c[0], np.asarray(c[1]).astype(int), np.asarray(c[2]), "cache"
        if mode == "readonly":
            raise FileNotFoundError(f"readonly: no cache for sub-{sid:02d} under {COV_DIR}.")
    raw, y, onsets = load_subject_raw(path)
    views = build_subject_covariances(raw, onsets)
    if mode in ("auto","rebuild"): save_subject_cov(sid, views, y, onsets)
    return views, y.astype(int), onsets, "computed"
print("Covariance builder + cache I/O defined.")

# 5. Paper pipeline (Steps 3-5: covariance -> LTSA -> classifier) + leaky/honest

In [ ]:
# ---- Steps 3-5: covariance -> TangentSpace -> (PCA) -> LTSA -> LDA   |   or FgMDM ----
def view_fit(cov_tr, y_tr):
    if CONFIG["twfb_classifier"] == "fgmdm":
        return {"kind":"fgmdm", "clf": FgMDM(metric=CONFIG["twfb_metric"]).fit(cov_tr, y_tr)}
    ts = TangentSpace(metric=CONFIG["twfb_metric"]).fit(cov_tr); Z = ts.transform(cov_tr); pca = None
    if CONFIG.get("ltsa_pre_pca") and Z.shape[1] > CONFIG["ltsa_pre_pca"]:
        nP = min(CONFIG["ltsa_pre_pca"], Z.shape[0]-1)
        pca = PCA(n_components=nP, random_state=CONFIG["random_state"]).fit(Z); Z = pca.transform(Z)
    ltsa = None; Zr = Z
    if CONFIG["use_ltsa"]:
        k = min(CONFIG["ltsa_n_neighbors"], Z.shape[0]-1)
        d = min(CONFIG["ltsa_n_components"], max(1,k-1), Z.shape[1])
        try:
            ltsa = LocallyLinearEmbedding(method="ltsa", n_neighbors=k, n_components=d, reg=CONFIG["ltsa_reg"],
                                          eigen_solver="dense", random_state=CONFIG["random_state"]).fit(Z)
            Zr = ltsa.transform(Z)
        except Exception:
            ltsa = None; Zr = Z      # safe fallback to (pca'd) tangent features if LTSA is unstable on a fold
    lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(Zr, y_tr)
    return {"kind":"lda_ltsa", "ts":ts, "pca":pca, "ltsa":ltsa, "lda":lda}

def view_pred(m, cov):
    if m["kind"] == "fgmdm": return m["clf"].predict(cov)
    Z = m["ts"].transform(cov)
    if m["pca"] is not None: Z = m["pca"].transform(Z)
    if m["ltsa"] is not None:
        try: Z = m["ltsa"].transform(Z)
        except Exception: pass
    return m["lda"].predict(Z)

def _score(cov_tr, y_tr, cov_te, y_te):
    try:
        m = view_fit(cov_tr, y_tr); pred = view_pred(m, cov_te)
        return float(accuracy_score(y_te, pred)), float(balanced_accuracy_score(y_te, pred))
    except Exception:
        return np.nan, np.nan

def _bacc(cov_tr, y_tr, cov_te, y_te): return _score(cov_tr, y_tr, cov_te, y_te)[1]

def leaky_oracle(views, y, n_repeats, rng):
    """LEAKY: per view a fresh 24/16 split, FULL pipeline fit, report max-over-views TEST accuracy."""
    n = len(y); tr_n, te_n = CONFIG["train_size"], CONFIG["test_size"]; reps = []
    for _ in range(n_repeats):
        accs = []
        for cov in views.values():
            p = rng.permutation(n); tr = p[:tr_n]; te = p[tr_n:tr_n+te_n]
            accs.append(_score(cov[tr], y[tr], cov[te], y[te])[0])
        reps.append(np.nanmax(accs))
    return float(np.nanmean(reps))

def honest_nested(views, y, splits, inner_folds, rng_state):
    """HONEST: select view by inner-CV bal.acc on TRAIN only (LTSA fit inside each inner fold), eval once on test."""
    keys = list(views.keys()); accs, baccs = [], []
    for tr, te in splits:
        skf = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=rng_state)
        best_key, best_v = keys[0], -np.inf
        for k in keys:
            cov = views[k]
            iv = [_bacc(cov[tr[a]], y[tr[a]], cov[tr[b]], y[tr[b]]) for a, b in skf.split(tr, y[tr])]
            mv = np.nanmean(iv)
            if mv > best_v: best_v, best_key = mv, k
        a, ba = _score(views[best_key][tr], y[tr], views[best_key][te], y[te])
        accs.append(a); baccs.append(ba)
    return float(np.nanmean(accs)), float(np.nanmean(baccs))
print("Paper pipeline (TangentSpace -> LTSA -> LDA) + leaky/honest protocols defined.")

# 6. Run all subjects

In [ ]:
def make_splits(y):
    idx = np.arange(len(y))
    if CONFIG["cv_scheme"] == "kfold5":
        sp = StratifiedKFold(n_splits=CONFIG["n_splits"], shuffle=True, random_state=CONFIG["random_state"])
    else:
        sp = StratifiedShuffleSplit(n_splits=CONFIG["n_repeats"], test_size=CONFIG["honest_test_frac"],
                                    random_state=CONFIG["random_state"])
    return list(sp.split(idx, y))

def run_all():
    files = find_subject_files(CONFIG["source_roots"]); keep = CONFIG["subjects_to_use"]
    rows = []; t0 = time.time(); rng = np.random.RandomState(CONFIG["random_state"]); n_cache = n_comp = 0
    print("="*70)
    for path in files:
        sid = int(os.path.basename(path).split("-")[1][:2])
        if keep is not None and sid not in keep: continue
        views, y, onsets, source = get_subject_covariances(sid, path)
        n_cache += (source=="cache"); n_comp += (source=="computed")
        twfb = {k:v for k,v in views.items() if k.startswith("tw__")}
        fb   = {k:v for k,v in views.items() if k.startswith("fb__")}
        splits = make_splits(y)
        row = {"subject": sid, "n_trials": int(len(y)), "cov_source": source}
        if CONFIG["run_twfb_leaky"]:
            row["twfb_leaky"] = leaky_oracle(twfb, y, CONFIG["n_repeats"], rng)
        if CONFIG["run_twfb_honest"]:
            a, ba = honest_nested(twfb, y, splits, CONFIG["inner_folds"], 0)
            row["twfb_honest_acc"], row["twfb_honest_bacc"] = a, ba
        if CONFIG["run_perband_fixed"]:
            for k, cov in fb.items():
                row[f"fixed_{k.replace('fb__','')}"] = float(np.nanmean(
                    [_score(cov[tr], y[tr], cov[te], y[te])[1] for tr, te in splits]))
        rows.append(row)
        shown = [c for c in ["twfb_leaky","twfb_honest_bacc"] if c in row]
        print(f"  sub-{sid:02d} [{source:8s}]: " + "  ".join(f"{c.replace('_bacc','')}={row[c]*100:5.1f}" for c in shown)
              + f"   ({time.time()-t0:5.1f}s)")
    print("="*70); print(f"cov source: {n_cache} from cache, {n_comp} computed")
    return pd.DataFrame(rows)

RESULTS = run_all()
RESULTS.head()

# 7. Summary, artifacts, plot

In [ ]:
def _ms(df, col):
    v = df[col].dropna().values
    return [float(np.mean(v)), float(np.std(v, ddof=1) if len(v) > 1 else 0.0)]

PROTO_COLS = [("twfb_leaky","Paper TWFB+LTSA (leaky max-on-test)"),
              ("twfb_honest_bacc","Paper TWFB+LTSA honest nested-CV (bal.acc)")]
summary = {c:_ms(RESULTS,c) for c,_ in PROTO_COLS if c in RESULTS}

print("="*70); print(f"Paper-faithful TWFB-DGFMDM + LTSA — Liu2024, n={len(RESULTS)}  [clf={CONFIG['twfb_classifier']} use_ltsa={CONFIG['use_ltsa']}]")
for c,label in PROTO_COLS:
    if c in summary: print(f"  {label:44s}: {summary[c][0]*100:5.2f}%  ± {summary[c][1]*100:4.2f}")
print(f"  {'Liu2024 Table 4 TWFB+DGFMDRM (reported)':44s}: 72.21%")
if "twfb_leaky" in summary and "twfb_honest_bacc" in summary:
    print(f"  >>> LEAKAGE GAP (leaky - honest): {(summary['twfb_leaky'][0]-summary['twfb_honest_bacc'][0])*100:.1f} pts")
if "twfb_honest_bacc" in RESULTS:
    h = RESULTS["twfb_honest_bacc"].dropna().values
    try: w,pw = sp_stats.wilcoxon(h-0.5)
    except Exception: pw=float("nan")
    print(f"  honest vs chance: mean={h.mean()*100:.2f}%  wilcoxon p={pw:.3g}  ({(h>0.5).sum()}/{len(h)} > 0.5)")
print("="*70)

RESULTS.to_csv(ARTIFACT_DIR/"subject_results.csv", index=False)
json.dump({"config":{k:(str(v) if isinstance(v,Path) else v) for k,v in CONFIG.items()}, "summary":summary,
           "n_subjects":int(len(RESULTS))}, open(ARTIFACT_DIR/"summary.json","w"), indent=2, default=str)
json.dump({"experiment_name":CONFIG["experiment_name"],"n_subjects":int(len(RESULTS)),"cov_signature":COV_SIG,
           "cov_cache_dir":str(COV_DIR),**summary}, open(ARTIFACT_DIR/"global_metrics.json","w"), indent=2, default=str)
json.dump(RESULTS.to_dict(orient="records"), open(ARTIFACT_DIR/"subject_metrics.json","w"), indent=2, default=str)
json.dump({"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"cov_cache_dir":str(COV_DIR),"cov_signature":COV_SIG,
           "config":CONFIG,"n_subjects":int(len(RESULTS))}, open(ARTIFACT_DIR/"run_metadata.json","w"), indent=2, default=str)

if HAVE_MPL and summary:
    cols=[c for c,_ in PROTO_COLS if c in summary]; vals=[summary[c][0]*100 for c in cols]; errs=[summary[c][1]*100 for c in cols]
    fig,ax=plt.subplots(figsize=(6,4)); ax.bar(range(len(cols)),vals,yerr=errs,capsize=5,color=["#c44","#2a7"][:len(cols)])
    ax.axhline(72.21,ls="--",c="purple",label="Liu Table 4 (72.21%)"); ax.axhline(50,ls=":",c="gray",label="chance")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([c.replace('_bacc','') for c in cols]); ax.set_ylabel("acc / bal.acc (%)")
    ax.set_title("Paper TWFB+LTSA: leaky vs honest"); ax.legend(fontsize=8)
    for i,v in enumerate(vals): ax.text(i,v+1,f"{v:.1f}",ha="center")
    plt.tight_layout(); plt.savefig(ARTIFACT_DIR/"leaky_vs_honest.png",dpi=200,bbox_inches="tight"); plt.close(fig)
print("Saved subject_results.csv, summary.json, global_metrics.json, subject_metrics.json, run_metadata.json + PNG")
print(f"Artifacts: {ARTIFACT_DIR}")